In [ ]:
from openai import OpenAI

client = OpenAI(api_key="0", base_url="http://192.168.106.55:20000/v1")

def llm_qwen(
    prompt, 
    model="Qwen3.5-27B",
    max_tokens=8192,
    temperature=0.7,
    top_p=0.95
):
    chat_response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
    )
    return chat_response.choices[0].message.content

if __name__ == "__main__":
    llm_output = llm_qwen("你知道什么是Tetlock吗？")
    print(llm_output)

In [1]:
# -*- coding: utf-8 -*-
"""
Superforecast MVP - JSON 本地存储版 / Jupyter 单 cell 版

使用方式：
1. 直接把整个代码块复制到一个 ipynb cell 里运行。
2. 修改下面 CONFIG / QUESTION / USER_EVIDENCE。
3. 执行：
   record = run_forecast(QUESTION, USER_EVIDENCE, CONFIG)
   print(record["markdown_report"])

常用操作：
- list_forecasts(CONFIG)
- show_forecast("forecast_id", CONFIG)
- resolve_forecast("forecast_id", outcome=1, config=CONFIG)
- export_forecast("forecast_id", out_path="report.md", config=CONFIG, fmt="md")
"""

from __future__ import annotations

import json
import math
import re
import uuid
from datetime import date, datetime
from pathlib import Path
from statistics import mean, pstdev
from typing import Any, Dict, List, Tuple, Optional

from openai import OpenAI


# =============================================================================
# 0. 所有可调参数都在这里
# =============================================================================

CONFIG = {
    # -------------------------
    # LLM 服务配置
    # -------------------------
    "api_key": "0",
    "base_url": "http://192.168.106.55:20000/v1",
    "model": "Qwen3.5-27B",

    # -------------------------
    # 本地 JSON 存储
    # -------------------------
    "storage_path": "./forecast_mvp_store.json",

    # -------------------------
    # LLM 通用参数
    # -------------------------
    "llm": {
        "max_tokens": 8192,
        "top_p": 0.95,
        "system": "You are a rigorous superforecasting research assistant. Output exactly what the user asks for.",
    },

    # -------------------------
    # 不同阶段的 temperature
    # -------------------------
    "temperature": {
        "contract": 0.10,
        "decomposition": 0.25,
        "evidence": 0.20,
        "panel": 0.55,
        "json_repair": 0.00,
    },

    # -------------------------
    # 预测引擎参数
    # -------------------------
    "engine": {
        # 概率边界，避免 logit 爆炸
        "min_probability": 0.01,
        "max_probability": 0.99,

        # base rate 边界
        "min_base_probability": 0.02,
        "max_base_probability": 0.98,

        # 单条证据裸 log-odds 影响边界
        "min_evidence_impact_log_odds": -0.80,
        "max_evidence_impact_log_odds": 0.80,

        # 单个因子 effect_log_odds 边界
        "min_factor_effect_log_odds": -1.50,
        "max_factor_effect_log_odds": 1.50,

        # 最多保留多少个因子 / 证据卡 / panel agent
        "max_factors": 12,
        "max_evidence_cards": 12,
        "max_panel_agents": 8,

        # 同源证据惩罚：第 n 条同组证据乘以 1 / sqrt(n)
        "independence_penalty_power": 0.5,

        # ensemble 权重公式
        # f = forecastability_score
        # base_weight = base_const + base_low_forecastability_bonus * (1 - f)
        # evidence_weight = evidence_const + evidence_forecastability_bonus * f
        # causal_weight = causal_const + causal_forecastability_bonus * f
        # panel_weight = panel_const
        "ensemble_weights": {
            "base_const": 0.20,
            "base_low_forecastability_bonus": 0.25,
            "evidence_const": 0.10,
            "evidence_forecastability_bonus": 0.30,
            "causal_const": 0.10,
            "causal_forecastability_bonus": 0.25,
            "panel_const": 0.20,
        },

        # 冷启动校准：没有足够历史样本时，向 50% 收缩
        "cold_start_min_history": 10,
        "cold_start_temperature_base": 1.25,
        "cold_start_temperature_forecastability_discount": 0.15,

        # 有历史数据后的 Brier 校准温度
        "brier_temperature": {
            "bad_threshold": 0.25,
            "weak_threshold": 0.22,
            "strong_threshold": 0.17,
            "bad_temperature": 1.45,
            "weak_temperature": 1.25,
            "normal_temperature": 1.12,
            "strong_temperature": 1.00,
        },

        # 置信区间参数，在 logit 空间扩张
        "interval": {
            "base_sigma": 0.55,
            "forecastability_penalty": 1.15,
            "panel_disagreement_weight": 0.30,
            "evidence_quality_penalty": 0.65,
            "evidence_count_bonus_per_card": 0.03,
            "max_evidence_count_bonus": 0.25,
            "min_sigma": 0.35,
            "max_sigma": 2.00,
        },
    },

    # -------------------------
    # 输出控制
    # -------------------------
    "runtime": {
        "print_steps": True,
        "print_markdown": True,
        "print_raw_json": False,
    },
}





# =============================================================================
# 1. LLM Client
# =============================================================================

client = OpenAI(
    api_key=CONFIG["api_key"],
    base_url=CONFIG["base_url"],
)


def llm_qwen(
    prompt: str,
    config: Dict[str, Any] = CONFIG,
    temperature: float = 0.30,
    max_tokens: Optional[int] = None,
    top_p: Optional[float] = None,
    system: Optional[str] = None,
) -> str:
    max_tokens = max_tokens or config["llm"]["max_tokens"]
    top_p = top_p if top_p is not None else config["llm"]["top_p"]
    system = system or config["llm"]["system"]

    chat_response = client.chat.completions.create(
        model=config["model"],
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
    )
    return chat_response.choices[0].message.content or ""


# =============================================================================
# 2. 基础工具函数
# =============================================================================

def now_iso() -> str:
    return datetime.now().replace(microsecond=0).isoformat()


def short_id() -> str:
    return uuid.uuid4().hex[:12]


def clamp(x: Any, lo: float, hi: float) -> float:
    try:
        x = float(x)
    except Exception:
        x = 0.0
    return max(lo, min(hi, x))


def safe_float(x: Any, default: float = 0.0) -> float:
    try:
        if x is None:
            return default
        return float(x)
    except Exception:
        return default


def logit(p: float) -> float:
    p = clamp(p, 1e-5, 1 - 1e-5)
    return math.log(p / (1 - p))


def sigmoid(x: float) -> float:
    if x >= 0:
        z = math.exp(-x)
        return 1.0 / (1.0 + z)
    z = math.exp(x)
    return z / (1.0 + z)


def probability_to_percent(p: float) -> str:
    return f"{float(p) * 100:.1f}%"


def json_dumps(obj: Any) -> str:
    return json.dumps(obj, ensure_ascii=False, indent=2)


def strip_think_blocks(text: str) -> str:
    return re.sub(
        r"<think>.*?</think>",
        "",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    ).strip()


def extract_json_object(text: str) -> Dict[str, Any]:
    text = strip_think_blocks(text).strip()

    fence = re.search(
        r"```(?:json)?\s*(.*?)\s*```",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    )
    if fence:
        text = fence.group(1).strip()

    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
        return {"value": obj}
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        candidate = text[start:end + 1]
        try:
            obj = json.loads(candidate)
            if isinstance(obj, dict):
                return obj
            return {"value": obj}
        except Exception as e:
            raise ValueError(f"JSON 解析失败：{e}\n\n原始输出前2000字符：\n{text[:2000]}")

    raise ValueError(f"没有找到 JSON 对象。\n\n原始输出前2000字符：\n{text[:2000]}")


def llm_json(
    prompt: str,
    config: Dict[str, Any] = CONFIG,
    temperature: float = 0.20,
    max_tokens: Optional[int] = None,
) -> Dict[str, Any]:
    strict_prompt = prompt.strip() + """

硬性要求：
1. 只输出一个合法 JSON 对象。
2. 不要输出 Markdown。
3. 不要输出解释。
4. 不要输出代码块。
5. 所有字符串必须使用双引号。
"""
    raw = llm_qwen(
        strict_prompt,
        config=config,
        temperature=temperature,
        max_tokens=max_tokens,
    )

    try:
        return extract_json_object(raw)
    except Exception:
        repair_prompt = f"""
下面是一段模型输出。请把它修复成一个合法 JSON 对象。

要求：
1. 只输出 JSON。
2. 不要解释。
3. 不要 Markdown。
4. 不要代码块。

原始输出：
{raw}
"""
        repaired = llm_qwen(
            repair_prompt,
            config=config,
            temperature=config["temperature"]["json_repair"],
            max_tokens=max_tokens,
        )
        return extract_json_object(repaired)


# =============================================================================
# 3. JSON 本地存储
# =============================================================================

def empty_store() -> Dict[str, Any]:
    return {
        "meta": {
            "name": "superforecast_mvp_json_store",
            "version": "0.1",
            "created_at": now_iso(),
            "updated_at": now_iso(),
        },
        "forecasts": [],
    }


def load_store(config: Dict[str, Any] = CONFIG) -> Dict[str, Any]:
    path = Path(config["storage_path"])
    if not path.exists():
        return empty_store()

    try:
        data = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(data, dict):
            return empty_store()
        if "forecasts" not in data:
            data["forecasts"] = []
        if "meta" not in data:
            data["meta"] = {}
        return data
    except Exception:
        backup = path.with_suffix(path.suffix + f".broken.{datetime.now().strftime('%Y%m%d_%H%M%S')}")
        path.rename(backup)
        print(f"原 JSON 文件解析失败，已备份到：{backup}")
        return empty_store()


def save_store(store: Dict[str, Any], config: Dict[str, Any] = CONFIG) -> None:
    path = Path(config["storage_path"])
    path.parent.mkdir(parents=True, exist_ok=True)

    store.setdefault("meta", {})
    store["meta"]["updated_at"] = now_iso()

    tmp_path = path.with_suffix(path.suffix + ".tmp")
    tmp_path.write_text(json.dumps(store, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp_path.replace(path)


def save_forecast_record(record: Dict[str, Any], config: Dict[str, Any] = CONFIG) -> None:
    store = load_store(config)
    store.setdefault("forecasts", [])
    store["forecasts"].append(record)
    save_store(store, config)


def find_forecast(forecast_id: str, config: Dict[str, Any] = CONFIG) -> Optional[Dict[str, Any]]:
    store = load_store(config)
    for item in store.get("forecasts", []):
        if item.get("id") == forecast_id:
            return item
    return None


def update_forecast_record(forecast_id: str, patch: Dict[str, Any], config: Dict[str, Any] = CONFIG) -> Dict[str, Any]:
    store = load_store(config)
    for idx, item in enumerate(store.get("forecasts", [])):
        if item.get("id") == forecast_id:
            item.update(patch)
            item["updated_at"] = now_iso()
            store["forecasts"][idx] = item
            save_store(store, config)
            return item
    raise ValueError(f"forecast_id 不存在：{forecast_id}")


def local_domain_brier_stats(domain: str, config: Dict[str, Any] = CONFIG) -> Dict[str, Any]:
    store = load_store(config)
    scores = []
    for item in store.get("forecasts", []):
        if (
            item.get("status") == "resolved"
            and item.get("contract", {}).get("domain", "other") == domain
            and item.get("brier_score") is not None
        ):
            scores.append(float(item["brier_score"]))

    if not scores:
        return {
            "n": 0,
            "avg_brier": None,
        }

    return {
        "n": len(scores),
        "avg_brier": mean(scores),
    }


# =============================================================================
# 4. Prompt Schema
# =============================================================================

CONTRACT_SCHEMA = {
    "normalized_question": "可结算的二元预测题，用一句话表达。必须包含截止日期或时间窗口。",
    "target_type": "binary",
    "deadline": "YYYY-MM-DD；如果用户未给日期，给出合理截止日期并在 ambiguity_flags 说明。",
    "resolution_criteria": "如何判定发生/未发生。必须可验证。",
    "resolution_sources": ["优先官方来源，其次主流媒体/数据库。不要编造URL。"],
    "domain": "AI | finance | geopolitics | tech_product | science | business | social | other",
    "forecastability_score": "0到1，表示努力预测是否有意义。",
    "ambiguity_flags": ["模糊点列表；没有则空数组。"],
    "clarifying_assumptions": ["系统为了继续预测而做出的最小假设。"],
    "reason": "简短说明。",
}

DECOMPOSITION_SCHEMA = {
    "base_rate": {
        "reference_class": "最接近的参考类。",
        "base_probability": "0到1，作为先验概率。",
        "sample_size_guess": "如果没有硬数据，写 rough/unknown；不要假装精确。",
        "confidence": "0到1",
        "rationale": "为什么这个参考类合理。",
    },
    "factors": [
        {
            "factor": "影响因素名称",
            "sub_question": "可被单独判断的子问题",
            "direction_if_true": "increase | decrease | mixed",
            "probability": "该因素为真的概率，0到1",
            "effect_log_odds": "如果该因素为真，对主事件log-odds影响，-1.5到1.5",
            "confidence": "0到1",
            "dependencies": ["依赖的其他factor名称"],
            "rationale": "简短理由",
        }
    ],
    "known_unknowns": ["关键未知项"],
    "search_queries": ["如果未来接入搜索，应搜索的query"],
}

EVIDENCE_SCHEMA = {
    "evidence_cards": [
        {
            "claim": "证据主张。不能编造具体来源。",
            "source_type": "user_provided | background | official | news | paper | market | database | other",
            "source": "来源名称；如果没有真实来源，写 background_prior 或 user_input，不要写假URL。",
            "direction": "increase | decrease | neutral",
            "relevance": "0到1",
            "reliability": "0到1",
            "diagnosticity": "0到1，表示这条证据对预测是否有区分度",
            "freshness": "0到1",
            "independence_group": "同源证据用同一个group，避免重复计权",
            "impact_log_odds": "证据对主事件log-odds的裸影响，-0.8到0.8",
            "rationale": "为什么这么赋权",
        }
    ]
}

PANEL_SCHEMA = {
    "agent_forecasts": [
        {
            "agent_name": "outside_view | inside_view | skeptic | optimist | base_rate_guardian | domain_generalist",
            "probability": "0到1",
            "confidence": "0到1",
            "rationale": "简短理由",
            "strongest_update_trigger": "什么新证据会明显改变判断",
        }
    ]
}


# =============================================================================
# 5. Agent 调用
# =============================================================================

def build_contract(question: str, config: Dict[str, Any] = CONFIG) -> Dict[str, Any]:
    prompt = f"""
你是 Forecast Contract Designer。你的任务是把用户的自然语言问题改写成一个“可结算、可评分、二元”的预测题。

今天日期：{date.today().isoformat()}

用户原始问题：
{question}

要求：
1. MVP 只支持 binary 预测。如果用户问连续变量，你要把它转成一个合理阈值型 binary 问题。
2. 必须给 deadline 和 resolution_criteria。
3. 不要假装拥有实时搜索结果。
4. 如果用户没给时间，选择一个合理截止日期，并写入 clarifying_assumptions。
5. resolution_sources 不要编造具体 URL，只写来源类别或机构名称。
6. forecastability_score 不是置信度，而是“努力预测是否有意义”的评分。

输出 JSON schema：
{json_dumps(CONTRACT_SCHEMA)}
"""
    obj = llm_json(
        prompt,
        config=config,
        temperature=config["temperature"]["contract"],
    )

    obj["target_type"] = "binary"
    obj["forecastability_score"] = clamp(obj.get("forecastability_score", 0.5), 0.0, 1.0)
    obj.setdefault("normalized_question", question)
    obj.setdefault("deadline", "")
    obj.setdefault("resolution_criteria", "")
    obj.setdefault("resolution_sources", [])
    obj.setdefault("domain", "other")
    obj.setdefault("ambiguity_flags", [])
    obj.setdefault("clarifying_assumptions", [])
    obj.setdefault("reason", "")

    return obj


def decompose_question(
    question: str,
    contract: Dict[str, Any],
    config: Dict[str, Any] = CONFIG,
) -> Dict[str, Any]:
    prompt = f"""
你是 Superforecasting Decomposition Agent。使用 Tetlock 风格的方法拆解预测题：

方法：
1. outside view / base rate
2. inside view / 机制分析
3. Fermi decomposition
4. 反方视角
5. 关键未知项

预测合约：
{json_dumps(contract)}

原始问题：
{question}

要求：
1. base_probability 是先验，不是最终概率。
2. factors 的 effect_log_odds 是数学引擎后续会使用的数值；不要全部给极端值。
3. 明确 dependencies，避免假设所有子问题独立。
4. 如果没有硬数据，要在 rationale 里说明不确定性。
5. 不要编造具体统计数据。
6. search_queries 只是为以后接入搜索准备，不代表你已经搜索过。

输出 JSON schema：
{json_dumps(DECOMPOSITION_SCHEMA)}
"""
    obj = llm_json(
        prompt,
        config=config,
        temperature=config["temperature"]["decomposition"],
    )

    engine = config["engine"]

    base = obj.get("base_rate", {}) or {}
    base["base_probability"] = clamp(
        base.get("base_probability", 0.5),
        engine["min_base_probability"],
        engine["max_base_probability"],
    )
    base["confidence"] = clamp(base.get("confidence", 0.5), 0.0, 1.0)
    base.setdefault("reference_class", "")
    base.setdefault("sample_size_guess", "unknown")
    base.setdefault("rationale", "")
    obj["base_rate"] = base

    clean_factors = []
    for f in (obj.get("factors") or [])[:engine["max_factors"]]:
        f["probability"] = clamp(
            f.get("probability", 0.5),
            engine["min_base_probability"],
            engine["max_base_probability"],
        )
        f["effect_log_odds"] = clamp(
            f.get("effect_log_odds", 0.0),
            engine["min_factor_effect_log_odds"],
            engine["max_factor_effect_log_odds"],
        )
        f["confidence"] = clamp(f.get("confidence", 0.5), 0.0, 1.0)
        f.setdefault("factor", "")
        f.setdefault("sub_question", "")
        f.setdefault("direction_if_true", "mixed")
        f.setdefault("dependencies", [])
        f.setdefault("rationale", "")
        clean_factors.append(f)

    obj["factors"] = clean_factors
    obj.setdefault("known_unknowns", [])
    obj.setdefault("search_queries", [])

    return obj


def build_evidence_cards(
    question: str,
    contract: Dict[str, Any],
    decomposition: Dict[str, Any],
    user_evidence: str = "",
    config: Dict[str, Any] = CONFIG,
) -> Dict[str, Any]:
    if user_evidence.strip():
        evidence_mode = "用户提供了证据材料，请优先从用户材料中抽取证据卡。"
    else:
        evidence_mode = "用户没有提供外部证据。只能生成 background 类型的分析性证据卡，不得编造新闻、URL、论文或具体数据。"

    prompt = f"""
你是 Evidence Structuring Agent。你的任务不是写文章，而是把信息整理成可计算的 evidence_cards。

{evidence_mode}

预测合约：
{json_dumps(contract)}

问题拆解：
{json_dumps(decomposition)}

用户提供的证据材料：
{user_evidence if user_evidence.strip() else "<empty>"}

要求：
1. 证据卡数量 4 到 12 条。
2. 不要编造具体来源、URL、统计数据、新闻标题。
3. 如果没有真实来源，source_type 必须使用 background，source 写 background_prior。
4. 如果证据来自用户材料，source_type 写 user_provided，source 写 user_input。
5. impact_log_odds 是裸影响，后续程序会乘以 relevance/reliability/diagnosticity/freshness 等权重。
6. 多条同源或高度相关证据必须使用同一个 independence_group。
7. diagnosticity 代表证据的区分度；泛泛而谈的信息不要给高分。

输出 JSON schema：
{json_dumps(EVIDENCE_SCHEMA)}
"""
    obj = llm_json(
        prompt,
        config=config,
        temperature=config["temperature"]["evidence"],
    )

    engine = config["engine"]

    clean_cards = []
    for c in (obj.get("evidence_cards") or [])[:engine["max_evidence_cards"]]:
        c["relevance"] = clamp(c.get("relevance", 0.5), 0.0, 1.0)
        c["reliability"] = clamp(c.get("reliability", 0.5), 0.0, 1.0)
        c["diagnosticity"] = clamp(c.get("diagnosticity", 0.5), 0.0, 1.0)
        c["freshness"] = clamp(c.get("freshness", 0.5), 0.0, 1.0)
        c["impact_log_odds"] = clamp(
            c.get("impact_log_odds", 0.0),
            engine["min_evidence_impact_log_odds"],
            engine["max_evidence_impact_log_odds"],
        )
        c.setdefault("claim", "")
        c.setdefault("source_type", "background")
        c.setdefault("source", "background_prior")
        c.setdefault("direction", "neutral")
        c.setdefault("independence_group", "default")
        c.setdefault("rationale", "")
        clean_cards.append(c)

    obj["evidence_cards"] = clean_cards
    return obj


def run_forecaster_panel(
    question: str,
    contract: Dict[str, Any],
    decomposition: Dict[str, Any],
    evidence: Dict[str, Any],
    config: Dict[str, Any] = CONFIG,
) -> Dict[str, Any]:
    prompt = f"""
你是一个多智能体预测委员会。请让以下 6 个虚拟预测者独立给出概率：

1. outside_view：只重视参考类和历史基准。
2. inside_view：重视当前事件机制。
3. skeptic：专门寻找失败路径和反证。
4. optimist：专门寻找成功路径，但不能无脑乐观。
5. base_rate_guardian：防止过度偏离 base rate。
6. domain_generalist：综合判断。

重要：
这些概率不是最终答案。最终概率由 Python 数学引擎聚合。

预测合约：
{json_dumps(contract)}

拆解：
{json_dumps(decomposition)}

证据卡：
{json_dumps(evidence)}

要求：
1. 每个 agent 必须给 0 到 1 概率。
2. 不要互相抄同一个概率。
3. 给出 strongest_update_trigger。
4. 不要声称自己完成了实时搜索。

输出 JSON schema：
{json_dumps(PANEL_SCHEMA)}
"""
    obj = llm_json(
        prompt,
        config=config,
        temperature=config["temperature"]["panel"],
    )

    engine = config["engine"]

    clean_runs = []
    for r in (obj.get("agent_forecasts") or [])[:engine["max_panel_agents"]]:
        r["probability"] = clamp(
            r.get("probability", 0.5),
            engine["min_base_probability"],
            engine["max_base_probability"],
        )
        r["confidence"] = clamp(r.get("confidence", 0.5), 0.0, 1.0)
        r.setdefault("agent_name", "agent")
        r.setdefault("rationale", "")
        r.setdefault("strongest_update_trigger", "")
        clean_runs.append(r)

    obj["agent_forecasts"] = clean_runs
    return obj


# =============================================================================
# 6. 概率引擎
# =============================================================================

def weight_evidence_cards(
    cards: List[Dict[str, Any]],
    config: Dict[str, Any] = CONFIG,
) -> Tuple[List[Dict[str, Any]], float, float]:
    engine = config["engine"]
    group_counts: Dict[str, int] = {}
    weighted_cards: List[Dict[str, Any]] = []
    total_delta = 0.0
    qualities = []

    penalty_power = engine["independence_penalty_power"]

    for c in cards:
        group = str(c.get("independence_group", "default"))
        group_counts[group] = group_counts.get(group, 0) + 1
        occurrence = group_counts[group]

        independence_penalty = 1.0 / (occurrence ** penalty_power)

        relevance = clamp(c.get("relevance", 0.5), 0.0, 1.0)
        reliability = clamp(c.get("reliability", 0.5), 0.0, 1.0)
        diagnosticity = clamp(c.get("diagnosticity", 0.5), 0.0, 1.0)
        freshness = clamp(c.get("freshness", 0.5), 0.0, 1.0)

        impact = clamp(
            c.get("impact_log_odds", 0.0),
            engine["min_evidence_impact_log_odds"],
            engine["max_evidence_impact_log_odds"],
        )

        quality_weight = relevance * reliability * diagnosticity * freshness * independence_penalty
        weighted_impact = impact * quality_weight

        c2 = dict(c)
        c2["independence_penalty"] = independence_penalty
        c2["quality_weight"] = quality_weight
        c2["weighted_impact_log_odds"] = weighted_impact

        weighted_cards.append(c2)
        total_delta += weighted_impact
        qualities.append(quality_weight)

    avg_quality = mean(qualities) if qualities else 0.0
    return weighted_cards, total_delta, avg_quality


def compute_causal_probability(
    base_p: float,
    factors: List[Dict[str, Any]],
    config: Dict[str, Any] = CONFIG,
) -> float:
    engine = config["engine"]
    x = logit(base_p)

    for f in factors[:engine["max_factors"]]:
        p_factor = clamp(
            f.get("probability", 0.5),
            engine["min_base_probability"],
            engine["max_base_probability"],
        )
        effect = clamp(
            f.get("effect_log_odds", 0.0),
            engine["min_factor_effect_log_odds"],
            engine["max_factor_effect_log_odds"],
        )
        conf = clamp(f.get("confidence", 0.5), 0.0, 1.0)

        centered = (p_factor - 0.5) * 2.0
        x += centered * effect * conf

    return clamp(
        sigmoid(x),
        engine["min_probability"],
        engine["max_probability"],
    )


def compute_panel_probability(
    agent_runs: List[Dict[str, Any]],
    base_p: float,
    config: Dict[str, Any] = CONFIG,
) -> Tuple[float, float]:
    engine = config["engine"]

    ps = [
        clamp(
            r.get("probability", base_p),
            engine["min_base_probability"],
            engine["max_base_probability"],
        )
        for r in agent_runs
    ]

    if not ps:
        return base_p, 0.0

    logits_all = [logit(p) for p in ps]
    logits_trimmed = sorted(logits_all)

    if len(logits_trimmed) >= 5:
        logits_trimmed = logits_trimmed[1:-1]

    panel_logit = mean(logits_trimmed)
    disagreement = pstdev(logits_all) if len(logits_all) > 1 else 0.0

    return (
        clamp(sigmoid(panel_logit), engine["min_probability"], engine["max_probability"]),
        disagreement,
    )


def ensemble_probabilities(
    base_p: float,
    evidence_p: float,
    causal_p: float,
    panel_p: float,
    forecastability: float,
    config: Dict[str, Any] = CONFIG,
) -> Dict[str, Any]:
    engine = config["engine"]
    ew = engine["ensemble_weights"]
    f = clamp(forecastability, 0.0, 1.0)

    weights = {
        "base": ew["base_const"] + ew["base_low_forecastability_bonus"] * (1.0 - f),
        "evidence": ew["evidence_const"] + ew["evidence_forecastability_bonus"] * f,
        "causal": ew["causal_const"] + ew["causal_forecastability_bonus"] * f,
        "panel": ew["panel_const"],
    }

    total = sum(weights.values())
    weights = {k: v / total for k, v in weights.items()}

    x = (
        weights["base"] * logit(base_p)
        + weights["evidence"] * logit(evidence_p)
        + weights["causal"] * logit(causal_p)
        + weights["panel"] * logit(panel_p)
    )

    p = clamp(
        sigmoid(x),
        engine["min_probability"],
        engine["max_probability"],
    )

    return {
        "probability": p,
        "weights": weights,
    }


def calibrate_probability(
    raw_p: float,
    forecastability: float,
    local_stats: Dict[str, Any],
    config: Dict[str, Any] = CONFIG,
) -> Tuple[float, Dict[str, Any]]:
    engine = config["engine"]

    n = int(local_stats.get("n") or 0)
    avg_brier = local_stats.get("avg_brier")
    f = clamp(forecastability, 0.0, 1.0)

    if n < engine["cold_start_min_history"] or avg_brier is None:
        temperature = (
            engine["cold_start_temperature_base"]
            - engine["cold_start_temperature_forecastability_discount"] * f
        )
        mode = "cold_start_conservative_shrinkage"
    else:
        bt = engine["brier_temperature"]
        avg_brier = float(avg_brier)

        if avg_brier > bt["bad_threshold"]:
            temperature = bt["bad_temperature"]
        elif avg_brier > bt["weak_threshold"]:
            temperature = bt["weak_temperature"]
        elif avg_brier < bt["strong_threshold"]:
            temperature = bt["strong_temperature"]
        else:
            temperature = bt["normal_temperature"]

        mode = "local_brier_temperature"

    calibrated = sigmoid(logit(raw_p) / temperature)

    meta = {
        "calibration_mode": mode,
        "temperature": temperature,
        "domain_history_n": n,
        "domain_avg_brier": avg_brier,
    }

    return (
        clamp(calibrated, engine["min_probability"], engine["max_probability"]),
        meta,
    )


def confidence_interval(
    p: float,
    forecastability: float,
    panel_disagreement: float,
    avg_evidence_quality: float,
    evidence_count: int,
    config: Dict[str, Any] = CONFIG,
) -> Tuple[float, float, Dict[str, Any]]:
    engine = config["engine"]
    interval = engine["interval"]

    f = clamp(forecastability, 0.0, 1.0)
    q = clamp(avg_evidence_quality, 0.0, 1.0)

    n_bonus = min(
        interval["max_evidence_count_bonus"],
        interval["evidence_count_bonus_per_card"] * max(0, evidence_count - 4),
    )

    sigma = (
        interval["base_sigma"]
        + interval["forecastability_penalty"] * (1.0 - f)
        + interval["panel_disagreement_weight"] * min(panel_disagreement, 2.0)
        + interval["evidence_quality_penalty"] * (1.0 - q)
        - n_bonus
    )

    sigma = clamp(
        sigma,
        interval["min_sigma"],
        interval["max_sigma"],
    )

    lo = sigmoid(logit(p) - sigma)
    hi = sigmoid(logit(p) + sigma)

    return (
        clamp(lo, 0.0, 1.0),
        clamp(hi, 0.0, 1.0),
        {"logit_sigma": sigma},
    )


def compute_probability_bundle(
    contract: Dict[str, Any],
    decomposition: Dict[str, Any],
    evidence_obj: Dict[str, Any],
    panel_obj: Dict[str, Any],
    local_stats: Dict[str, Any],
    config: Dict[str, Any] = CONFIG,
) -> Dict[str, Any]:
    engine = config["engine"]

    base_p = clamp(
        (decomposition.get("base_rate") or {}).get("base_probability", 0.5),
        engine["min_base_probability"],
        engine["max_base_probability"],
    )

    forecastability = clamp(
        contract.get("forecastability_score", 0.5),
        0.0,
        1.0,
    )

    evidence_cards, total_evidence_delta, avg_quality = weight_evidence_cards(
        evidence_obj.get("evidence_cards") or [],
        config=config,
    )

    evidence_p = clamp(
        sigmoid(logit(base_p) + total_evidence_delta),
        engine["min_probability"],
        engine["max_probability"],
    )

    causal_p = compute_causal_probability(
        base_p,
        decomposition.get("factors") or [],
        config=config,
    )

    panel_p, disagreement = compute_panel_probability(
        panel_obj.get("agent_forecasts") or [],
        base_p,
        config=config,
    )

    ensemble = ensemble_probabilities(
        base_p,
        evidence_p,
        causal_p,
        panel_p,
        forecastability,
        config=config,
    )

    raw_p = ensemble["probability"]

    calibrated_p, calibration_meta = calibrate_probability(
        raw_p,
        forecastability,
        local_stats,
        config=config,
    )

    low, high, interval_meta = confidence_interval(
        calibrated_p,
        forecastability,
        disagreement,
        avg_quality,
        len(evidence_cards),
        config=config,
    )

    return {
        "base_probability": base_p,
        "evidence_probability": evidence_p,
        "causal_probability": causal_p,
        "panel_probability": panel_p,
        "raw_probability": raw_p,
        "calibrated_probability": calibrated_p,
        "confidence_low": low,
        "confidence_high": high,
        "weighted_evidence_cards": evidence_cards,
        "total_evidence_delta_log_odds": total_evidence_delta,
        "avg_evidence_quality": avg_quality,
        "panel_disagreement_logit_std": disagreement,
        "ensemble_weights": ensemble["weights"],
        "calibration": calibration_meta,
        "interval": interval_meta,
    }


# =============================================================================
# 7. 报告生成
# =============================================================================

def top_evidence(cards: List[Dict[str, Any]], k: int = 6) -> List[Dict[str, Any]]:
    return sorted(
        cards,
        key=lambda c: abs(safe_float(c.get("weighted_impact_log_odds"), 0.0)),
        reverse=True,
    )[:k]


def build_report_json(
    forecast_id: str,
    original_question: str,
    contract: Dict[str, Any],
    decomposition: Dict[str, Any],
    evidence_obj: Dict[str, Any],
    panel_obj: Dict[str, Any],
    prob: Dict[str, Any],
    config: Dict[str, Any] = CONFIG,
) -> Dict[str, Any]:
    return {
        "id": forecast_id,
        "created_at": now_iso(),
        "updated_at": now_iso(),
        "status": "open",
        "outcome": None,
        "resolved_at": None,
        "brier_score": None,
        "original_question": original_question,
        "contract": contract,
        "decomposition": decomposition,
        "evidence": {
            **evidence_obj,
            "evidence_cards": prob["weighted_evidence_cards"],
        },
        "panel": panel_obj,
        "probability_engine": {
            k: v
            for k, v in prob.items()
            if k not in {"weighted_evidence_cards"}
        },
        "runtime_config_snapshot": {
            "model": config["model"],
            "base_url": config["base_url"],
            "engine": config["engine"],
            "temperature": config["temperature"],
        },
    }


def build_markdown_report(report: Dict[str, Any]) -> str:
    contract = report["contract"]
    decomp = report["decomposition"]
    engine = report["probability_engine"]
    cards = report["evidence"]["evidence_cards"]
    panel = report.get("panel", {}).get("agent_forecasts", [])

    p = float(engine["calibrated_probability"])
    low = float(engine["confidence_low"])
    high = float(engine["confidence_high"])

    base_rate = decomp.get("base_rate", {}) or {}
    factors = decomp.get("factors", []) or []
    known_unknowns = decomp.get("known_unknowns", []) or []
    ambiguity_flags = contract.get("ambiguity_flags", []) or []
    assumptions = contract.get("clarifying_assumptions", []) or []

    lines = []

    lines.append(f"# Forecast {report['id']}")
    lines.append("")
    lines.append(f"**预测题**：{contract.get('normalized_question', '')}")
    lines.append(f"**最终概率**：{probability_to_percent(p)}")
    lines.append(f"**粗略区间**：{probability_to_percent(low)} - {probability_to_percent(high)}")
    lines.append(f"**截止日期**：{contract.get('deadline', '')}")
    lines.append(f"**领域**：{contract.get('domain', 'other')}")
    lines.append(f"**可预测性评分**：{safe_float(contract.get('forecastability_score'), 0.5):.2f}")
    lines.append("")

    lines.append("## 判定标准")
    lines.append(str(contract.get("resolution_criteria", "")))
    sources = contract.get("resolution_sources", []) or []
    if sources:
        lines.append(f"判定来源：{', '.join(map(str, sources))}")
    lines.append("")

    lines.append("## 概率计算拆解")
    lines.append(f"- Base rate prior：{probability_to_percent(engine['base_probability'])}")
    lines.append(f"- Evidence update 后：{probability_to_percent(engine['evidence_probability'])}")
    lines.append(f"- Causal factor 聚合后：{probability_to_percent(engine['causal_probability'])}")
    lines.append(f"- 多 Agent panel 聚合后：{probability_to_percent(engine['panel_probability'])}")
    lines.append(f"- Raw ensemble：{probability_to_percent(engine['raw_probability'])}")
    lines.append(f"- Calibrated final：{probability_to_percent(engine['calibrated_probability'])}")
    lines.append("")

    lines.append("## Ensemble 权重")
    weights = engine.get("ensemble_weights", {})
    for k, v in weights.items():
        lines.append(f"- {k}: {float(v):.3f}")
    lines.append("")

    lines.append("## 参考类 / Base Rate")
    lines.append(f"参考类：{base_rate.get('reference_class', '')}")
    lines.append(f"先验概率：{probability_to_percent(safe_float(base_rate.get('base_probability'), 0.5))}")
    lines.append(f"置信度：{safe_float(base_rate.get('confidence'), 0.5):.2f}")
    lines.append(f"说明：{base_rate.get('rationale', '')}")
    lines.append("")

    lines.append("## 关键因子")
    if factors:
        for f in factors[:8]:
            lines.append(
                f"- **{f.get('factor', '')}**："
                f"P={probability_to_percent(safe_float(f.get('probability'), 0.5))}，"
                f"effect_log_odds={safe_float(f.get('effect_log_odds'), 0.0):+.2f}，"
                f"方向={f.get('direction_if_true', '')}。"
                f"{f.get('rationale', '')}"
            )
    else:
        lines.append("- 无结构化因子。")
    lines.append("")

    lines.append("## 最有诊断价值的证据")
    top_cards = top_evidence(cards, k=6)
    if top_cards:
        for c in top_cards:
            lines.append(
                f"- [{c.get('direction', 'neutral')}] {c.get('claim', '')} "
                f"| weighted_log_odds={safe_float(c.get('weighted_impact_log_odds'), 0.0):+.3f} "
                f"| quality_weight={safe_float(c.get('quality_weight'), 0.0):.3f} "
                f"| source={c.get('source_type', '')}/{c.get('source', '')}"
            )
            rat = str(c.get("rationale", "")).strip()
            if rat:
                lines.append(f"  - {rat}")
    else:
        lines.append("- 无证据卡。")
    lines.append("")

    lines.append("## 多 Agent 分歧")
    if panel:
        for r in panel:
            lines.append(
                f"- **{r.get('agent_name', 'agent')}**："
                f"{probability_to_percent(safe_float(r.get('probability'), 0.5))}，"
                f"confidence={safe_float(r.get('confidence'), 0.5):.2f}。"
                f"{r.get('rationale', '')}"
            )
            trigger = str(r.get("strongest_update_trigger", "")).strip()
            if trigger:
                lines.append(f"  - 更新触发器：{trigger}")
    else:
        lines.append("- 无 panel 结果。")
    lines.append("")

    if known_unknowns:
        lines.append("## 关键未知项")
        for x in known_unknowns[:8]:
            lines.append(f"- {x}")
        lines.append("")

    if ambiguity_flags or assumptions:
        lines.append("## 模糊点与假设")
        for x in ambiguity_flags[:8]:
            lines.append(f"- 模糊点：{x}")
        for x in assumptions[:8]:
            lines.append(f"- 假设：{x}")
        lines.append("")

    cal = engine.get("calibration", {})
    lines.append("## 校准信息")
    lines.append(f"- calibration_mode：{cal.get('calibration_mode')}")
    lines.append(f"- temperature：{safe_float(cal.get('temperature'), 1.0):.3f}")
    lines.append(f"- domain_history_n：{cal.get('domain_history_n')}")
    lines.append(f"- domain_avg_brier：{cal.get('domain_avg_brier')}")
    lines.append("")

    lines.append("## 结论")
    lines.append(
        f"当前给出的概率是 **{probability_to_percent(p)}**。"
        f"这个数字由 base rate、证据 log-odds、因子聚合、多 Agent panel 和本地校准共同计算得到。"
    )

    return "\n".join(lines).strip() + "\n"


# =============================================================================
# 8. 主流程
# =============================================================================

def run_forecast(
    question: str,
    user_evidence: str = "",
    config: Dict[str, Any] = CONFIG,
) -> Dict[str, Any]:
    runtime = config["runtime"]

    def step(msg: str):
        if runtime.get("print_steps", True):
            print(msg)

    step("[1/5] 生成预测合约...")
    contract = build_contract(question, config=config)
    print("问题改写:",json.dumps(contract,ensure_ascii=False, indent=2))
    step("[2/5] 拆解问题、生成参考类和因子...")
    decomposition = decompose_question(question, contract, config=config)
    print("问题拆解:",json.dumps(decomposition,ensure_ascii=False, indent=2))
    step("[3/5] 结构化证据卡...")
    evidence_obj = build_evidence_cards(
        question,
        contract,
        decomposition,
        user_evidence=user_evidence,
        config=config,
    )
    print("证据卡:",json.dumps(evidence_obj,ensure_ascii=False, indent=2))
    step("[4/5] 运行多 Agent 预测委员会...")
    panel_obj = run_forecaster_panel(
        question,
        contract,
        decomposition,
        evidence_obj,
        config=config,
    )
    print("Panel 结果:",json.dumps(panel_obj,ensure_ascii=False, indent=2))
    step("[5/5] 概率聚合、校准、本地 JSON 落库...")
    domain = str(contract.get("domain", "other"))
    stats = local_domain_brier_stats(domain, config=config)
    prob = compute_probability_bundle(
        contract,
        decomposition,
        evidence_obj,
        panel_obj,
        stats,
        config=config,
    )

    forecast_id = short_id()

    report_json = build_report_json(
        forecast_id,
        question,
        contract,
        decomposition,
        evidence_obj,
        panel_obj,
        prob,
        config=config,
    )

    markdown_report = build_markdown_report(report_json)

    record = {
        **report_json,
        "markdown_report": markdown_report,
    }

    save_forecast_record(record, config=config)

    if runtime.get("print_markdown", True):
        print("\n" + markdown_report)

    if runtime.get("print_raw_json", False):
        print("\n--- RAW JSON ---")
        print(json.dumps(record, ensure_ascii=False, indent=2))

    return record


# =============================================================================
# 9. 本地管理函数
# =============================================================================

def list_forecasts(config: Dict[str, Any] = CONFIG, limit: int = 20) -> List[Dict[str, Any]]:
    store = load_store(config)
    items = store.get("forecasts", [])
    items = sorted(items, key=lambda x: x.get("created_at", ""), reverse=True)[:limit]

    for item in items:
        engine = item.get("probability_engine", {})
        p = engine.get("calibrated_probability")
        p_str = probability_to_percent(p) if p is not None else "n/a"

        status = item.get("status", "open")
        brier = item.get("brier_score")
        brier_str = f", brier={float(brier):.4f}" if brier is not None else ""

        question = item.get("contract", {}).get("normalized_question", item.get("original_question", ""))
        print(f'{item.get("id")} | {item.get("created_at")} | {p_str} | {status}{brier_str} | {question}')

    return items


def show_forecast(
    forecast_id: str,
    config: Dict[str, Any] = CONFIG,
    raw_json: bool = False,
) -> Dict[str, Any]:
    item = find_forecast(forecast_id, config=config)
    if item is None:
        raise ValueError(f"forecast_id 不存在：{forecast_id}")

    print(item.get("markdown_report", ""))

    if raw_json:
        print("\n--- RAW JSON ---")
        print(json.dumps(item, ensure_ascii=False, indent=2))

    return item


def resolve_forecast(
    forecast_id: str,
    outcome: int,
    config: Dict[str, Any] = CONFIG,
    force: bool = False,
) -> Dict[str, Any]:
    if outcome not in (0, 1):
        raise ValueError("outcome 只能是 0 或 1")

    item = find_forecast(forecast_id, config=config)
    if item is None:
        raise ValueError(f"forecast_id 不存在：{forecast_id}")

    if item.get("status") == "resolved" and not force:
        raise ValueError("该预测已经结算。如果要覆盖，设置 force=True。")

    p = float(item.get("probability_engine", {}).get("calibrated_probability", 0.5))
    brier = (p - outcome) ** 2

    patch = {
        "status": "resolved",
        "outcome": outcome,
        "resolved_at": now_iso(),
        "brier_score": brier,
    }

    updated = update_forecast_record(forecast_id, patch, config=config)

    print(f"Resolved {forecast_id}: outcome={outcome}, p={p:.4f}, brier={brier:.4f}")
    return updated


def export_forecast(
    forecast_id: str,
    out_path: str,
    config: Dict[str, Any] = CONFIG,
    fmt: str = "md",
) -> str:
    item = find_forecast(forecast_id, config=config)
    if item is None:
        raise ValueError(f"forecast_id 不存在：{forecast_id}")

    path = Path(out_path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if fmt == "json":
        path.write_text(json.dumps(item, ensure_ascii=False, indent=2), encoding="utf-8")
    elif fmt == "md":
        path.write_text(item.get("markdown_report", ""), encoding="utf-8")
    else:
        raise ValueError("fmt 只能是 md 或 json")

    print(f"已导出：{path}")
    return str(path)


# =============================================================================
# 10. 直接运行示例
# =============================================================================
# 你在 ipynb 里主要改这里
QUESTION = """
明年中国北京的经济是否会比今年更好？
"""

USER_EVIDENCE = """

""".strip()

record = run_forecast(QUESTION, USER_EVIDENCE, CONFIG)

[1/5] 生成预测合约...
问题改写: {
  "normalized_question": "2027年中国北京市的地区生产总值（GDP）是否将高于2026年？",
  "target_type": "binary",
  "deadline": "2028-03-31",
  "resolution_criteria": "若北京市统计局或国家统计局发布的2027年全年北京市GDP最终核实数高于2026年全年北京市GDP最终核实数，则判定为“是”；否则判定为“否”。",
  "resolution_sources": [
    "北京市统计局",
    "国家统计局",
    "北京市国民经济和社会发展统计公报"
  ],
  "domain": "finance",
  "forecastability_score": 0.7,
  "ambiguity_flags": [
    "用户未指定“更好”的具体指标，默认采用GDP作为核心经济指标",
    "用户未指定具体截止日期，默认以2027年全年的最终统计数据为准"
  ],
  "clarifying_assumptions": [
    "假设“经济更好”等同于名义GDP或实际GDP的增长，此处采用名义GDP数值比较以避免通缩/通胀定义的复杂性，通常官方发布时会提供可比价增速，若可比价增速为正即视为增长",
    "假设数据发布延迟不超过2028年3月31日"
  ],
  "reason": "将模糊的“经济更好”转化为可量化的GDP同比变化，并设定了合理的数据发布截止日期。"
}
[2/5] 拆解问题、生成参考类和因子...
问题拆解: {
  "base_rate": {
    "reference_class": "中国主要城市（直辖市/省会）年度名义GDP同比增长率",
    "base_probability": 0.85,
    "sample_size_guess": "rough",
    "confidence": 0.7,
    "rationale": "在过去几十年中，中国经济整体保持正增长，北京作为首都和核心经济引擎，其名义GDP（包含价格因素）几乎每年都在增长。除非发生严重的系统性经济危机或通缩，否则名义GDP下降的概率极低。即使实际GDP增速放